In [1]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


## Generating a large dataset (50 million+ rows) using pandas and numpy

## Installing Polars and Pandera

In [1]:
pip install polars pandera

Note: you may need to restart the kernel to use updated packages.


## Step 1: Defining the Contract (Schema Validation)

In [2]:
import polars as pl
import pandera.polars as pa

# Define the strict schema contract for the cleaned transactions
transaction_schema = pa.DataFrameSchema({
    "transaction_id": pa.Column(str),
    "customer_id": pa.Column(str),
    "amount": pa.Column(float, checks=pa.Check.ge(0.0)),
    "currency": pa.Column(str, checks=pa.Check.isin(["USD", "EUR", "GBP"])),
    "timestamp": pa.Column(pl.Datetime(time_zone="UTC")),
    "transaction_type": pa.Column(str, checks=pa.Check.isin(["credit", "debit"])),
    "status": pa.Column(str, nullable=False)
})

## Step 2: Lazy Loading & Deduplication

In [3]:
# Create LazyFrames (Query plans, no data is loaded into memory yet)
customer_lazy = pl.scan_csv('customers_raw.csv')
transactions_lazy = pl.scan_csv('transactions_raw.csv')

# PHASE 1: Deduplication
customers_clean = customer_lazy.unique(subset=["customer_id"])
transactions_clean = transactions_lazy.unique(subset=["transaction_id"])

## Step 3: Expression-Based Normalization & Math

In [4]:
# PHASE 2 & 4: Text Normalization and Numeric Validation using Expressions
customers_clean = customers_clean.with_columns([
    pl.col("country_code").str.strip_chars().str.to_uppercase(),
    pl.col("kyc_status").str.strip_chars().str.to_uppercase(),
    # PHASE 5: Handle Nulls on the fly
    pl.col("first_name").fill_null("NOT_PROVIDED"),
    pl.col("email").fill_null("NOT_PROVIDED")
])

transactions_clean = transactions_clean.with_columns([
    pl.col("currency").str.strip_chars().str.to_uppercase(),
    pl.col("amount").abs()
]).drop_nulls(subset=["status"])


## Step 4: Temporal Standardization (The Fast Way)

In [5]:
# PHASE 3: Temporal Standardization
# Polars is strict. If the format varies wildly, we use coalesce to try multiple formats.

customers_clean = customers_clean.with_columns([
    pl.coalesce([
        pl.col("join_date").str.to_datetime("%Y-%m-%d", strict=False),
        pl.col("join_date").str.to_datetime("%m/%d/%Y", strict=False),
        pl.col("join_date").str.to_datetime("%d-%m-%Y", strict=False),
    ]).alias("join_date")
])

# For ISO timestamps, Polars parses them natively and fast
# We use replace_time_zone to ensure everything is localized to UTC
transactions_clean = transactions_clean.with_columns([
    pl.col("timestamp")
    .str.to_datetime(strict=False)
    .dt.replace_time_zone("UTC", ambiguous="earliest")
])

## Step 5: Execution and Validation

In [7]:
# Execute the query plan across all CPU cores
df_transaction_final = transactions_clean.collect()
df_customers_final = customers_clean.collect()

# Validate the final DataFrame against the Pandera schema
try:
    df_transactions_validated = transaction_schema.validate(df_transaction_final)
    print("Success: Large-scale data cleaned and validated!")
except pa.errors.SchemaError as e:
    print(f"Validation failed! Bad data caught before hitting the database: {e}")

Validation failed! Bad data caught before hitting the database: non-nullable column 'timestamp' contains null values
